# Part 1 - 실습 1: Amazon Bedrock 기초 & Foundation Model 탐색
**소요시간: 50분** | 난이도: ⭐⭐

## 학습 목표
1. Amazon Bedrock 클라이언트를 초기화하고 사용 가능한 Foundation Model 목록을 조회합니다.
2. `invoke_model` API로 Claude에게 첫 번째 프롬프트를 전송합니다.
3. `converse` API(통합 인터페이스)로 멀티턴 대화를 구현합니다.

## Amazon Bedrock 아키텍처
```
사용자 코드
   │
   ├─ bedrock          (모델 목록 조회)
   ├─ bedrock-runtime  (텍스트 생성, 스트리밍)
   └─ bedrock-agent-runtime  (RAG / Knowledge Base)
```

## 주요 모델 ID (2025 기준)
```
anthropic.claude-3-5-sonnet-20241022-v2:0  ← 최신 고성능
anthropic.claude-3-haiku-20240307-v1:0    ← 빠르고 경제적
amazon.titan-text-premier-v1:0            ← AWS 자체 모델
meta.llama3-8b-instruct-v1:0              ← 오픈소스 계열
```


---
## 🏢 기업 시나리오 — 생성형 AI 도입 TF

당신은 회사의 **생성형 AI 도입 TF 담당자**입니다.
"GPT 같은 걸 우리 업무에도 써보자"는 요청을 받았습니다. 모델을 직접 학습시키지 않고 **API 호출만으로** 시작합니다.

이번 실습에서는 다음을 수행합니다.
1. **모델 조사** → 어떤 Foundation Model이 있는지 목록 조회 (list_foundation_models)
2. **첫 호출** → Claude에게 프롬프트를 보내 응답 받기 (invoke_model)
3. **멀티턴 대화** → 챗봇의 토대가 되는 대화 구현 (converse)

> 💡 모델 선택은 비용·속도·품질의 trade-off입니다. 보통 저렴·빠른 모델(Haiku)로 PoC를 시작하고, 품질이 부족하면 상위 모델로 올립니다.


In [12]:
# ✅ [제공 코드] 환경 초기화
import boto3, json
import pandas as pd

# Bedrock 클라이언트 (모델 목록 조회용)
bedrock = boto3.client('bedrock', region_name='us-east-1')

# Bedrock Runtime 클라이언트 (텍스트 생성용)
bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-1')

print('✅ Bedrock 클라이언트 생성 완료')
print(f'  bedrock           : {bedrock.meta.service_model.service_name}')
print(f'  bedrock-runtime   : {bedrock_runtime.meta.service_model.service_name}')


✅ Bedrock 클라이언트 생성 완료
  bedrock           : bedrock
  bedrock-runtime   : bedrock-runtime


## ✏️ TODO 1: Foundation Model 목록 조회

Bedrock에서 사용 가능한 Foundation Model 목록을 조회하고 텍스트 생성 모델만 필터링하세요.


In [13]:
# ✏️ TODO 1: list_foundation_models API로 사용 가능한 모델을 조회하세요
response = bedrock.list_foundation_models(
    byOutputModality='TEXT'    # ← 'TEXT'
)

models = response['modelSummaries']   # ← 'modelSummaries'
print(f'텍스트 생성 모델 수: {len(models)}개\n')

rows = []
for m in models:
    rows.append({
        '모델ID'    : m['modelId'],                 # ← 'modelId'
        '제공사'    : m['providerName'],                 # ← 'providerName'
        '모델명'    : m['modelName'],
    })

df = pd.DataFrame(rows)
print(df[df['모델ID'].str.contains('claude|titan|llama')].to_string(index=False))


텍스트 생성 모델 수: 93개

                                        모델ID       제공사                           모델명
     anthropic.claude-sonnet-4-20250514-v1:0 Anthropic               Claude Sonnet 4
    anthropic.claude-haiku-4-5-20251001-v1:0 Anthropic              Claude Haiku 4.5
                    anthropic.claude-fable-5 Anthropic                Claude Fable 5
                 anthropic.claude-sonnet-4-6 Anthropic             Claude Sonnet 4.6
                anthropic.claude-opus-4-6-v1 Anthropic               Claude Opus 4.6
                   anthropic.claude-opus-4-8 Anthropic               Claude Opus 4.8
                   anthropic.claude-opus-4-7 Anthropic               Claude Opus 4.7
   anthropic.claude-sonnet-4-5-20250929-v1:0 Anthropic             Claude Sonnet 4.5
     anthropic.claude-opus-4-1-20250805-v1:0 Anthropic               Claude Opus 4.1
     anthropic.claude-opus-4-5-20251101-v1:0 Anthropic               Claude Opus 4.5
 anthropic.claude-3-sonnet-20240229-v1:0:28k An

## ✏️ TODO 2: invoke_model — 첫 번째 텍스트 생성

Claude에게 프롬프트를 전송하고 응답을 파싱하세요. `invoke_model`은 모델별 고유 페이로드 구조를 사용합니다.

```python
# Claude 3 페이로드 구조
{
    'anthropic_version': 'bedrock-2023-05-31',
    'max_tokens': 1024,
    'messages': [{'role': 'user', 'content': '프롬프트 내용'}]
}
```


In [14]:
# ✏️ TODO 2: invoke_model API로 Claude에게 질문을 전송하세요
# us-east-1이면 프리픽스는 us. 입니다.
MODEL_ID = 'us.anthropic.claude-sonnet-4-20250514-v1:0'

payload = {
    'anthropic_version': 'bedrock-2023-05-31',
    'max_tokens': 1024,           # ← 1024
    'messages': [
        {
            'role': 'user',         # ← 'user'
            'content': 'AWS Bedrock을 한 문장으로 설명해줘.'       # ← 'AWS Bedrock을 한 문장으로 설명해줘.'
        }
    ]
}

response = bedrock_runtime.invoke_model(
    modelId=MODEL_ID,                 # ← MODEL_ID
    body=json.dumps(payload),        # ← payload
    contentType='application/json'
)

result = json.loads(response['body'].read())
answer = result['content'][0]['text']   # ← 'content', 'text'
print('Claude 응답:')
print(answer)


Claude 응답:
AWS Bedrock은 Amazon이 제공하는 관리형 서비스로, 다양한 파운데이션 모델(Foundation Model)을 API를 통해 쉽게 사용할 수 있게 해주는 생성형 AI 플랫폼입니다.


In [15]:
# ✏️ TODO 2-2: invoke_model API로 Nova에게 질문을 전송하세요
MODEL_ID = 'amazon.nova-micro-v1:0'

payload = {
    'schemaVersion': 'messages-v1',    # Nova 전용 필드
    'inferenceConfig': {'maxTokens': 1024},  # 힌트: 응답으로 생성할 최대 토큰 수 (정수)
    'messages': [
        {
            'role': 'user',             # 힌트: 대화에서 사람 쪽 역할을 나타내는 문자열
            'content': [{'text': 'AWS Bedrock을 한 문장으로 설명해줘.'}]  # 힌트: 모델에게 보낼 프롬프트 문자열을 입력하세요
        }
    ]
}

response = bedrock_runtime.invoke_model(
    modelId=MODEL_ID,                     # 힌트: 이 셀 상단에 정의된 모델 ID 변수를 참조하세요
    body=json.dumps(payload),             # 힌트: 위에서 정의한 페이로드 딕셔너리를 JSON 직렬화해 전달하세요
    contentType='application/json'
)

result = json.loads(response['body'].read())
answer = result['output']['message']['content'][0]['text']
print('Nova 응답:')
print(answer)

Nova 응답:
AWS Bedrock은 고객이 고급 AI 모델을 쉽게 사용하고 맞춤화할 수 있도록 지원하는 아마존 웹 서비스의 기반 인프라입니다.


In [19]:
# ✅ [제공 코드] Temperature & Max Tokens 파라미터 실험
def call_claude(prompt, temperature=0.7, max_tokens=512):
    payload = {
        'anthropic_version': 'bedrock-2023-05-31',
        'max_tokens': max_tokens,
        'temperature': temperature,
        'messages': [{'role': 'user', 'content': prompt}]
    }
    resp = bedrock_runtime.invoke_model(
        modelId=MODEL_ID, body=json.dumps(payload), contentType='application/json'
    )
    return json.loads(resp['body'].read())['content'][0]['text']

prompt = '파이썬의 장점을 세 가지만 알려줘.'
for temp in [0.1, 0.7, 1.2]:
    print(f'\n--- temperature={temp} ---')
    print(call_claude(prompt, temperature=temp))



--- temperature=0.1 ---


파이썬의 주요 장점 3가지를 소개해드리겠습니다.

## 1. **간단하고 읽기 쉬운 문법**
- 영어와 유사한 직관적인 문법으로 코드 작성이 쉽습니다
- 들여쓰기로 코드 블록을 구분해 가독성이 뛰어납니다
- 초보자도 빠르게 학습할 수 있습니다

## 2. **풍부한 라이브러리 생태계**
- 웹 개발, 데이터 분석, AI/ML, 자동화 등 다양한 분야의 라이브러리가 풍부합니다
- pip를 통해 쉽게 외부 패키지를 설치하고 관리할 수 있습니다
- NumPy, Pandas, Django, Flask 등 검증된 라이브러리들이 많습니다

## 3. **높은 생산성과 개발 속도**
- 적은 코드로 많은 기능을 구현할 수 있습니다
- 인터프리터 언어로 컴파일 과정 없이 바로 실행 가능합니다
- 프로토타이핑과 빠른 개발에 매우 적합합니다

이러한 장점들로 인해 파이썬은 전 세계적으로 가장 인기 있는 프로그래밍 언어 중 하나가 되었습니다.

--- temperature=0.7 ---


파이썬의 주요 장점 3가지를 소개해드리겠습니다.

## 1. **간단하고 읽기 쉬운 문법**
- 영어와 유사한 직관적인 문법으로 코드 작성이 쉽습니다
- 들여쓰기로 코드 블록을 구분하여 가독성이 뛰어납니다
- 초보자도 빠르게 학습할 수 있습니다

## 2. **풍부한 라이브러리 생태계**
- 웹 개발, 데이터 분석, 머신러닝, 자동화 등 다양한 분야의 라이브러리가 풍부합니다
- pip를 통해 쉽게 외부 패키지를 설치하고 관리할 수 있습니다
- NumPy, Pandas, Django, Flask 등 검증된 라이브러리들이 많습니다

## 3. **높은 생산성과 개발 속도**
- 적은 코드로 많은 기능을 구현할 수 있습니다
- 프로토타입 개발이나 빠른 개발이 필요한 프로젝트에 적합합니다
- 크로스 플랫폼 지원으로 다양한 운영체제에서 동일하게 실행됩니다

이러한 장점들 덕분에 파이썬은 초보자부터 전문가까지 널리 사용되고 있습니다.

--- temperature=1.2 ---


ValidationException: An error occurred (ValidationException) when calling the InvokeModel operation: temperature: range: 0..1

## ✏️ TODO 3: converse API — 통합 멀티턴 대화

`converse`는 모델별 페이로드 차이를 추상화한 통합 API입니다. 대화 히스토리를 누적하여 멀티턴을 구현하세요.

```python
# converse 구조
bedrock_runtime.converse(
    modelId='...',
    messages=[{'role': 'user', 'content': [{'text': '...'}]}],
    inferenceConfig={'maxTokens': 512, 'temperature': 0.7}
)
```


In [20]:
# ✏️ TODO 3: converse API로 멀티턴 대화를 구현하세요
conversation_history = []

def chat(user_message, system_prompt=None):
    """대화 히스토리를 유지하며 converse API를 호출합니다"""
    conversation_history.append({
        'role': 'user',                           # ← 'user'
        'content': [{'text': user_message}]             # ← user_message
    })

    kwargs = {
        'modelId': MODEL_ID,
        'messages': conversation_history,                       # ← conversation_history
        'inferenceConfig': {'maxTokens': 512, 'temperature': 0.7}
    }
    if system_prompt:
        kwargs['system'] = [{'text': system_prompt}]

    response = bedrock_runtime.converse(**kwargs)

    assistant_msg = response['output']['message']['content'][0]['text']  # ← 'message', 'text'
    conversation_history.append({
        'role': 'assistant',
        'content': [{'text': assistant_msg}]
    })
    return assistant_msg

# 멀티턴 대화 테스트
SYSTEM = '당신은 친절한 AI 어시스턴트입니다. 한국어로 답변하세요.'

print('사용자:', '안녕하세요! 클라우드 컴퓨팅이 뭔가요?')
print('AI:', chat('안녕하세요! 클라우드 컴퓨팅이 뭔가요?', SYSTEM))
print()
print('사용자:', 'AWS와 Azure 중 어떤 게 더 좋아요?')
print('AI:', chat('AWS와 Azure 중 어떤 게 더 좋아요?'))
print()
print(f'대화 히스토리 길이: {len(conversation_history)}개 메시지')


사용자: 안녕하세요! 클라우드 컴퓨팅이 뭔가요?


AI: 안녕하세요! 클라우드 컴퓨팅에 대해 쉽게 설명해드릴게요.

**클라우드 컴퓨팅**은 인터넷을 통해 컴퓨터 자원(서버, 저장공간, 소프트웨어 등)을 필요할 때마다 빌려 쓸 수 있는 서비스입니다.

## 주요 특징
- **언제 어디서나 접근**: 인터넷만 있으면 어디서든 이용 가능
- **필요한 만큼만 사용**: 사용량에 따라 비용 지불
- **유지보수 불필요**: 서비스 제공업체가 관리

## 일상 속 클라우드 서비스 예시
- **구글 드라이브, 네이버 클라우드**: 파일 저장
- **넷플릭스, 유튜브**: 동영상 스트리밍  
- **카카오톡, 인스타그램**: 메시지, 사진 동기화

## 장점
✅ 초기 투자 비용 절약
✅ 자동 백업 및 보안
✅ 필요에 따른 확장/축소 가능
✅ 최신 기술 자동 업데이트

쉽게 말해서, 컴퓨터를 직접 사지 않고도 인터넷으로 빌려쓰는 것이라고 생각하시면 됩니다!

더 궁금한 점이 있으시면 언제든 물어보세요.

사용자: AWS와 Azure 중 어떤 게 더 좋아요?


AI: 좋은 질문이에요! AWS와 Azure는 각각 장단점이 있어서 "어떤 상황에서" 더 좋은지로 말씀드리는 게 정확할 것 같아요.

## 🏆 AWS (Amazon Web Services)
**강점:**
- 시장 점유율 1위 (가장 많이 사용됨)
- 서비스 종류가 가장 다양함 (200개 이상)
- 풍부한 학습 자료와 커뮤니티
- 스타트업부터 대기업까지 폭넓은 사용층

**추천 상황:**
- 처음 클라우드를 배우는 경우
- 다양한 서비스 실험이 필요한 경우
- 글로벌 서비스를 운영하는 경우

## 🏆 Microsoft Azure
**강점:**
- Windows/Office와의 뛰어난 연동성
- 기업용 소프트웨어와 자연스러운 통합
- 하이브리드 클라우드에 강함
- 한국어 지원이 상대적으로 좋음

**추천 상황:**
- 회사에서 Microsoft 제품을 많이 사용하는 경우
- 기존 온프레미스 시스템과 연결이 필요한 경우
- 엔터프라이즈 환경에서 작업하는 경우

## 💡 결론
- **학습용/개인 프로젝트**: AWS 추천
- **Microsoft 생태계 기업**: Azure 추천
- **비용**: 사용 패턴에 따라 다름

어떤 용도로 사용하실 계획인지 알려주시면 더

대화 히스토리 길이: 4개 메시지


---
## 🔗 실무로 연결하기

`사용자 입력` → `Lambda` → `Bedrock(converse)` → `응답` 의 흐름이 모든 생성형 AI 앱의 기본 골격입니다.

- **모델 선택 기준**: Haiku(빠름·저렴, 대량 처리) / Sonnet·Opus(고품질, 복잡 추론). 비용은 토큰 사용량 기반.
- PoC는 저렴한 모델로 시작 → 품질 검증 후 상위 모델 또는 프롬프트 개선.
- `converse` API는 모델이 바뀌어도 코드를 거의 그대로 재사용할 수 있어 실무에서 선호됩니다.


## 💡 심화 도전
1. `meta.llama3-8b-instruct-v1:0`을 `converse`로 호출하여 Claude와 같은 질문에 대한 응답을 비교해보세요.
2. Temperature를 0.01로 설정했을 때와 1.5로 설정했을 때 동일 프롬프트 결과가 어떻게 달라지는지 확인하세요.
3. `converse`의 `system` 파라미터로 다른 성격의 AI 페르소나를 설정해보세요.


## ✅ 정답 코드

👆 모두 풀고 난 후 확인하세요

```python
# TODO 1
response = bedrock.list_foundation_models(byOutputModality='TEXT')
models = response['modelSummaries']
m['modelId'], m['providerName']

# TODO 2
payload = {
    'anthropic_version': 'bedrock-2023-05-31',
    'max_tokens': 1024,
    'messages': [{'role': 'user', 'content': 'AWS Bedrock을 한 문장으로 설명해줘.'}]
}
response = bedrock_runtime.invoke_model(modelId=MODEL_ID, body=json.dumps(payload), ...)
answer = result['content'][0]['text']

# TODO 3
conversation_history.append({'role': 'user', 'content': [{'text': user_message}]})
kwargs = {'modelId': MODEL_ID, 'messages': conversation_history, ...}
assistant_msg = response['output']['message']['content'][0]['text']
```
